In [1]:
import pandas as pd, json, re, os, textwrap, collections, numpy as np
from pathlib import Path
base=Path('/workspace')
print('Analysis plan: 1) independently rescore all RSI titles by instrument themes; 2) identify recurring/lead-or-last authors; 3) audit pre-screen abstracts; 4) screen manuscript references and BYU/name conflicts; 5) verify current affiliations/activity on the web; 6) rank for five-pillar coverage.')
tsv=pd.read_csv(base/'rsi_all_articles_compact.tsv', sep='\t')
rel=pd.read_json(base/'rsi_relevant_articles.jsonl', lines=True)
rank=pd.read_csv(base/'rsi_author_ranking.csv')
text=(base/'paper_text.txt').read_text(errors='replace')
print('Shapes:',tsv.shape, rel.shape, rank.shape, 'manuscript chars',len(text))
print(tsv.head(2).to_string(index=False))
print('\nMissing values:',tsv.isna().sum().to_dict())
print('Year range/count:',tsv.year.min(),tsv.year.max(),tsv.year.value_counts().sort_index().to_dict())

Analysis plan: 1) independently rescore all RSI titles by instrument themes; 2) identify recurring/lead-or-last authors; 3) audit pre-screen abstracts; 4) screen manuscript references and BYU/name conflicts; 5) verify current affiliations/activity on the web; 6) rank for five-pillar coverage.
Shapes: (11869, 4) (351, 7) (2305, 6) manuscript chars 47210
              doi  year                                   authors                                                                                                                                    title
10.1063/1.4994651  2018 Xia W; Ma Y; Han S; Wang Y; Liu F; Zhai Y                 A new three-dimensional nonscanning laser imaging system based on the illumination pattern of a point-light-source array
10.1063/5.0203542  2024                Zhang W; Lin M; Li H; Qi X Finite element analysis of pulse sharpening effect of gyromagnetic nonlinear transmission line based on Landau–Lifshitz–Gilbert equation

Missing values: {'doi': 0, 'year': 

In [2]:
# Independent thematic title sweep over full corpus
patterns={
'induction':r'\b(induction|inductive|radio.?frequency heating|rf heating|electromagnetic levitat)',
'furnace':r'\b(furnace|microfurnace|anneal|thermal stage|heating stage|heater|high.temperature cell|sample environment)',
'pyrometry':r'\b(pyromet|radiation thermometr|emissivit|spectral radiance|infrared thermometr|temperature measur)',
'processing':r'\b(levitat|containerless|floating zone|zone melt|solidification|crystal growth|sinter|laser heating|thermal analysis)',
'vacuum_gas':r'\b(vacuum|ultra.high vacuum|uhv|controlled atmosphere|gas.loading|gas integrated)',
'ebsd':r'\b(ebsd|electron backscatter|grain growth|microstructure mapping|serial section)',
'open_control':r'\b(open.source|labview|data acquisition|computer.control|automated|automation|feedback|controller)'
}
for k,p in patterns.items(): tsv[k]=tsv.title.str.contains(p,case=False,regex=True,na=False)
tsv['n_theme']=tsv[list(patterns)].sum(axis=1)
hits=tsv[tsv[list(patterns)].any(axis=1)].copy()
print('Independent sweep hits:',len(hits), 'of',len(tsv))
print(hits.sort_values(['n_theme','year'],ascending=[False,False])[['year','doi','title']].head(40).to_string(index=False))

# Author table preserving positions
rows=[]
for _,r in tsv.iterrows():
    aa=[] if pd.isna(r.authors) else [x.strip() for x in str(r.authors).split(';')]
    for i,a in enumerate(aa):
        rows.append((a,r.doi,int(r.year),r.title,i+1,len(aa), i==0, i==len(aa)-1, *[bool(r[k]) for k in patterns]))
cols=['author','doi','year','title','position','n_authors','first','last']+list(patterns)
adf=pd.DataFrame(rows,columns=cols)
thematic=adf[adf[list(patterns)].any(axis=1)]
agg=thematic.groupby('author').agg(n_theme_papers=('doi','nunique'), recent=('year',lambda x:(x>=2021).sum()), firstlast=('first', 'sum'), last=('last','sum'), latest=('year','max')).reset_index()
# firstlast actual OR
agg['lead_end']=thematic.groupby('author').apply(lambda g:(g['first']|g['last']).sum(),include_groups=False).values
print('\nTop recurring thematic authors:')
print(agg.sort_values(['n_theme_papers','lead_end','recent','latest'],ascending=False).head(60).to_string(index=False))

/tmp/ipykernel_23/3743114277.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for k,p in patterns.items(): tsv[k]=tsv.title.str.contains(p,case=False,regex=True,na=False)
/tmp/ipykernel_23/3743114277.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for k,p in patterns.items(): tsv[k]=tsv.title.str.contains(p,case=False,regex=True,na=False)
/tmp/ipykernel_23/3743114277.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for k,p in patterns.items(): tsv[k]=tsv.title.str.contains(p,case=False,regex=True,na=False)
/tmp/ipykernel_23/3743114277.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for k,p in patterns.items(): tsv[k]=t

Independent sweep hits: 737 of 11869
 year               doi                                                                                                                                                                                                                                                                                   title
 2026 10.1063/5.0299443                                                                                                                                                                                 Automated rapid cooling of high-temperature vacuum furnaces for high throughput neutron experimentation
 2016 10.1063/1.4954926                                                                                                                                                                                                                       Ultra-high vacuum compatible induction-heated rod casting furnace
 2026 10.1063/5.0292738                            


Top recurring thematic authors:
       author  n_theme_papers  recent  firstlast  last  latest  lead_end
      Zhang Y              22      17          4     2    2026         6
        Liu Y              18      10          6     5    2026        11
        Liu H              14       6          6     0    2025         6
       Wang X              14       9          2     2    2026         4
       Wang J              12       7          2     3    2024         5
       Wang H              12       6          1     4    2025         5
      Zhang X              12      10          2     2    2026         4
        Liu Z              12       3          0     1    2023         1
       Yang J              11       7          3     1    2026         4
       Wang Y              11       5          1     0    2025         1
       Wang Z              10       7          4     0    2025         4
       Yang X              10       7          1     1    2025         2
         Li Y     

In [3]:
# Pull exact corpus evidence for a broad candidate set and inspect full-name/affiliation metadata
candidate_tokens=['Kargl','Dreißigacker','Wilson','Zaefferer','Deep','Jagadeesh','Tai','Pfleiderer','Bauer','An','Mendonça','Mendonca','Le Maoult','Short','Robertson','Weber','Salamat','Zhang','Shu','Jiménez','Iuchi','Kabra','Lohöfer','Kelton','Niedziela']
sel=rel[rel.authors.apply(lambda aa:any(any(tok.lower() in ((a.get('given','')+' '+a.get('family','')).lower()) for tok in candidate_tokens) for a in aa))]
for _,r in sel.sort_values(['year','doi'],ascending=[False,True]).iterrows():
    names=[]
    for a in r.authors:
        nm=(a.get('given','')+' '+a.get('family','')).strip()
        if any(tok.lower() in nm.lower() for tok in candidate_tokens):
            aff=' | '.join(a.get('affiliation',[]) or [])
            names.append(f'{nm} [{aff}]')
    print(f"{r.year} {r.doi} | {r.title}\n  {'; '.join(names)}")

2026 10.1063/5.0293861 | A simple wire-coil resistive heater for high temperature radial x-ray diffraction in a diamond anvil cell
  J. Keane [Department of Chemistry & Biochemistry, University of California Los Angeles 3 , Los Angeles, California 90095,]
2026 10.1063/5.0295219 | A testing method for aeroengine casings under high-temperature aerodynamic fatigue loads
  Changjian Zhao [China Academy of Launch Vehicle Technology 1 , Beijing 100076,]; Feng Yang [China Academy of Launch Vehicle Technology 1 , Beijing 100076, | School of Aerospace Engineering, Huazhong University of Science and Technology 2 , Wuhan 430074,]; Xiaohui Wang [Beijing Institute of Structure and Environment Engineering 3 , Beijing 100076,]; Lian Shen [Aero Engine Corporation of China Sichuan Gas Turbine Research Institute 4 , Chengdu 610500,]; Jiang Zhu [China Academy of Launch Vehicle Technology 1 , Beijing 100076,]; Hongyan Shao [Aero Engine Corporation of China Sichuan Gas Turbine Research Institute 4 , Chengd

In [4]:
# Exact-name collision/coauthor audit within RSI corpus for manuscript authors
needles=['Baird S','Weber R','Nyborg C','Guymon R','Erickson G','Johnson O']
for n in needles:
    m=tsv[tsv.authors.fillna('').str.contains(r'(^|; )'+re.escape(n)+r'(;|$)',regex=True)]
    print('\n',n, len(m))
    if len(m): print(m[['year','doi','authors','title']].to_string(index=False))

# Extract reference section and search whether candidate surnames occur there
refs=text[text.lower().find('references'):]
for surname in ['Kargl','Dreissigacker','Dreißigacker','Wilson','Zaefferer','Deep','Jagadeesh','Tai','Pfleiderer','Bauer','An','Mendonça','Le Maoult','Short','Robertson','Salamat','Jiménez','Iuchi','Kabra','Lohöfer','Kelton','Niedziela','Dubrovinsky']:
    if re.search(r'\b'+re.escape(surname)+r'\b',refs,re.I): print('CITED MATCH',surname)
print('Reference section length',len(refs))


 Baird S 0

 Weber R 2
 year               doi                                                                   authors                                                                                                                                  title
 2019 10.1063/1.5055738                               McCormack S; Tamalonis A; Weber R; Kriven W Temperature gradients for thermophysical and thermochemical property measurements to 3000 °C for an aerodynamically levitated spheroid
 2023 10.1063/5.0148455 Boland S; Wilke S; Scott J; Schlossberg S; Ivaschenko A; Weber R; Lipke D                                                                A hyperbaric aerodynamic levitator for containerless materials research

 Nyborg C 0

 Guymon R 0

 Erickson G 0

 Johnson O 0
Reference section length 1


/tmp/ipykernel_23/4056833576.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m=tsv[tsv.authors.fillna('').str.contains(r'(^|; )'+re.escape(n)+r'(;|$)',regex=True)]
/tmp/ipykernel_23/4056833576.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m=tsv[tsv.authors.fillna('').str.contains(r'(^|; )'+re.escape(n)+r'(;|$)',regex=True)]


In [5]:
from e14c.web import web_search
queries=[
'Frank Kargl DLR current affiliation furnace solidification',
'Stephen D Wilson UCSB current faculty laser floating zone',
'Cheng-Chi Tai National Cheng Kung University professor induction heating',
'Christian Pfleiderer TUM current professor induction furnace',
'Andreas Bauer TUM current affiliation physicist',
'Gopalan Jagadeesh IISc current professor pyrometry',
'Jerome Mendonca microfurnace NewTec Scientific current',
'Stefan Zaefferer MPIE current group EBSD',
'Ke An Oak Ridge National Laboratory current',
'Yannick Le Maoult Institut Clement Ader current',
'S Jimenez LIFTEC pyrometry University Zaragoza full name',
'T Iuchi Toyo University radiation thermometry full name',
'Oliver K Johnson BYU collaborators Sterling Baird University Utah Toronto Acceleration Consortium',
]
results={}
for q in queries:
    try:
        results[q]=await web_search(q,num_results=4)
        print('\nQUERY',q)
        for x in results[q][:3]: print('-',x.get('title'),x.get('url'))
    except Exception as e: print('ERR',q,e)


QUERY Frank Kargl DLR current affiliation furnace solidification
- Prof. Dr. rer. nat. Florian Kargl https://rwthcontacts.rwth-aachen.de/person/PER-28PCTVW
- Chair Fundamentals of Solidification | GI | RWTH Aachen University | EN https://www.gi.rwth-aachen.de/cms/gi/das-institut/lehrstuehle/~xkmlc/lehrstuhl-grundlagen-der-erstarrung/?lidx=1
- Chair Fundamentals of Solidification | GI | RWTH Aachen University | EN https://www.gi.rwth-aachen.de/cms/gi/Das-Institut/Lehrstuehle/~xkmlc/Lehrstuhl-Grundlagen-der-Erstarrung/lidx/1/



QUERY Stephen D Wilson UCSB current faculty laser floating zone
- Stephen Wilson - UCSB Materials - UC Santa Barbara https://www.materials.ucsb.edu/people/faculty/stephen-wilson
- Stephen D. Wilson | Wilson Group | Materials Department | UC Santa Barbara https://labs.materials.ucsb.edu/wilson/stephen/members/wilson
- High-pressure, Laser-based Floating-Zone Crystal Growth https://qis.quantumfoundry.ucsb.edu/projects/high-pressure-laser-based-floating-zone-crystal-growth



QUERY Cheng-Chi Tai National Cheng Kung University professor induction heating
- Professor Cheng-Chi Tai https://www.ee.ncku.edu.tw/en/teacher/index2.php?teacher_id=125
- Cheng-Chi Tai
      -  National Cheng Kung University https://researchoutput.ncku.edu.tw/en/persons/cheng-chi-tai/
- A COMPACT HALF-BRIDGE INDUCTION HEATING SYSTEM FOR MAGNETIC NANOPARTICLE THERMOTHERAPY APPLICATIONS https://doi.org/10.1142/s1016237207000057



QUERY Christian Pfleiderer TUM current professor induction furnace
- Prof. Dr. Christian Pfleiderer https://www.professoren.tum.de/en/pfleiderer-christian
- Christian Pfleiderer https://inspirehep.net/authors/2031592
- Pfleiderer Group - Zentrum für QuantumEngineering https://www.zqe.tum.de/zqe/people/pfleiderer-group/



QUERY Andreas Bauer TUM current affiliation physicist
- Lehre - Chair of Experimental Physics on the Topology of Correlated Systems https://www.ph.nat.tum.de/en/tcs/lehre/
- ‪Andreas Bauer‬ - ‪Google Scholar‬ https://scholar.google.de/citations?hl=en&user=JhClx6sAAAAJ
- Andreas Bauer https://scholar.google.com/citations?hl=de&user=JhClx6sAAAAJ


...


QUERY Ke An Oak Ridge National Laboratory current
- Ke An | ORNL https://www.ornl.gov/staff-profile/ke
- Dr Ke AN https://web.ornl.gov/~k6e/ORNL/Ke_An.html
- Ke An https://linkedin.com/in/keanornl



QUERY Yannick Le Maoult Institut Clement Ader current
- yannick le maoult | Professeur Institut Mines Télécom (enseignant chercheur) -Ecoles des Mines d'Albi https://www.linkedin.com/in/yannick-le-maoult-04a62557
- Le Maoult, Yannick (19..-....) https://www.idref.fr/145217418
- Archive ouverte HAL https://cv.hal.science/yannick-le-maoult



QUERY S Jimenez LIFTEC pyrometry University Zaragoza full name
- Measurement of gas temperature around a burning droplet: thin filament and soot pyrometry | DIGITAL.CSIC https://digital.csic.es/handle/10261/357628
- Laboratorio de Investigación en Fluidodinámica y Tecnologías de la Combustión (LIFTEC) | Universidad de Zaragoza https://www.unizar.es/estructura/centros-de-investigacion/laboratorio-de-investigacion-en-fluidodinamica-y-tecnologias-de
- Personal - LIFTEC - UNIZAR/CSIC http://www.liftec.unizar-csic.es/es/25-instituto/personal



QUERY T Iuchi Toyo University radiation thermometry full name
- 井内 徹 | 研究者情報 | J-GLOBAL 科学技術総合リンクセンター https://jglobal.jst.go.jp/detail?JGLOBAL_ID=200901026266131680
- 井内 徹 (Tohru Iuchi) - MISC - researchmap https://researchmap.jp/read0027619/misc
- Tohru Iuchi https://exa.ai/library/person/43qf9ky6xvg17zx1mp251n140



QUERY Oliver K Johnson BYU collaborators Sterling Baird University Utah Toronto Acceleration Consortium
- Acceleration Consortium 2025 https://acceleration.utoronto.ca/people/sterling-baird
- People - Johnson Group - BYU https://johnson.byu.edu/content/people
- Papers https://johnson.byu.edu/papers


In [6]:
# Print all corpus records for named finalists, avoiding ambiguous surname-only matching where possible
final_tokens=['Kargl F','Dreißigacker C','Wilson S','Tai C','Bauer A','Pfleiderer C','An K','Mendonça J','Mendonca J','Zhang Y','Shu S','Pontillon Y','Gallais L','Lohöfer G','Lohofer G','Dubrovinsky L','Dubrovinskaia N','Salamat A','Kabra S','Zaefferer S','Jagadeesh G','Jiménez S','Jimenez S','Iuchi T','Weber S']
for tok in final_tokens:
    m=tsv[tsv.authors.fillna('').apply(lambda s: tok in [x.strip() for x in s.split(';')])]
    if len(m):
        print('\n###',tok,'n=',len(m))
        print(m.sort_values('year',ascending=False)[['year','doi','title']].to_string(index=False))


### Kargl F n= 5
 year               doi                                                                                                      title
 2023 10.1063/5.0151523             X-radiography front tracking gradient furnace for directional solidification of bulk Al-alloys
 2021 10.1063/5.0037398            Isothermal furnace for long-term in situ and real-time X-radiography solidification experiments
 2020 10.1063/5.0004356                                                    Gas-loading furnace for deuterium-charged alloy-casting
 2020 10.1063/1.5124548 XRISE-M: X-radiography facility for solidification and diffusion studies of alloys aboard sounding rockets
 2015 10.1063/1.4922359             Near-isothermal furnace for in situ and real time X-ray radiography solidification experiments

### Dreißigacker C n= 7
 year               doi                                                                                                      title
 2021 10.1063/5.0047346          An expe


### Dubrovinsky L n= 8
 year               doi                                                                                                                                                title
 2022 10.1063/5.0071786 Testing the performance of secondary anvils shaped with focused ion beam from the single-crystal diamond for use in double-stage diamond anvil cells
 2021 10.1063/5.0050190                                    Isothermal equation of state of crystalline and glassy materials from optical measurements in diamond anvil cells
 2019 10.1063/1.5128592                                                     Table-top nuclear magnetic resonance system for high-pressure studies with in situ laser heating
 2019 10.1063/1.5108881                    Single-crystal diffractometer coupled with double-sided laser heating system at the Extreme Conditions Beamline P02.2 at PETRAIII
 2019 10.1063/1.5117786                                Laser heating setup for diamond anvil cells for in situ 

In [7]:
# Compact exact evidence for remaining finalists, using full names from enriched JSONL to avoid initials collisions
families={'Yuzhong Zhang','Shuangbao Shu','Y. Pontillon','Laurent Gallais','Ke An','G. Lohöfer','Sebastian Weber','Ashkan Salamat','Stefan Zaefferer'}
for _,r in rel.sort_values(['year','doi'],ascending=[False,True]).iterrows():
    matched=[]
    for i,a in enumerate(r.authors):
        nm=(a.get('given','')+' '+a.get('family','')).strip()
        if any((f.lower()==nm.lower()) or (f.split()[-1].lower()==a.get('family','').lower() and f.split()[0].lower().rstrip('.') in a.get('given','').lower()) for f in families):
            matched.append((nm,i+1,len(r.authors),' | '.join(a.get('affiliation',[]) or [])))
    if matched:
        print(f"{r.year} {r.doi} | {r.title}\n  {matched}")

2026 10.1063/5.0299443 | Automated rapid cooling of high-temperature vacuum furnaces for high throughput neutron experimentation
  [('Ke An', 7, 11, 'Oak Ridge National Laboratory 2 , Oak Ridge, Tennessee 37830,')]
2025 10.1063/5.0202933 | A laser-based annealing methodology to speed-up the study of thermo-activated restoration mechanisms in metals
  [('Laurent Gallais', 12, 12, 'Aix-Marseille Univ, CNRS, Centrale Med, Institut Fresnel 3 , Marseille,')]
2024 10.1063/5.0182719 | TEMPUS—A microgravity electromagnetic levitation facility for parabolic flights
  [('G. Lohöfer', 1, 7, 'Institut für Materialphysik im Weltraum, Deutsches Zentrum für Luft- und Raumfahrt (DLR) , 51170 Köln, Germany')]
2023 10.1063/5.0139508 | A laser-based system to heat nuclear fuel pellets at high temperature
  [('Y. Pontillon', 8, 9, 'CEA, DES, IRESNE, DEC, Cadarache 1 , F-13108 Saint-Paul-Lez-Durance, France')]
2022 10.1063/5.0087945 | Development of a new, fully automated system for electron backscatter di

In [8]:
# Find linked paper series and portfolios needed to ensure 2-4 corpus-supported records per recommended candidate
terms=['microfurnace dedicated','laser-based annealing','high power laser facility','laser-based system to heat nuclear','electromagnetic levitation facility','inductive measurement of electrical resistivity','CCD-based pyrometer','spectral band emissivity','CO2 laser heating system','thermo-mechanical rig','controlled atmosphere','vacuum furnace','electrostatic levitation facility']
for term in terms:
    m=tsv[tsv.title.str.contains(term,case=False,regex=False)]
    print('\nTERM',term)
    print(m[['year','doi','authors','title']].to_string(index=False))


TERM microfurnace dedicated
 year               doi                                                     authors                                                                                                                                                   title
 2024 10.1063/5.0207477 Mendonça J; Lautru J; Brau H; Nogues D; Candeias A; Podor R    Development of a microfurnace dedicated to in situ scanning electron microscope observation up to 1300 °C. III. In situ high temperature experiments
 2024 10.1063/5.0207475           Mendonça J; Brau H; Nogues D; Candeias A; Podor R Development of a microfurnace dedicated to in situ scanning electron microscope observation up to 1300 °C. II. Study of the thermal response of samples
 2024 10.1063/5.0207466           Mendonça J; Brau H; Nogues D; Candeias A; Podor R      Development of a microfurnace dedicated to in situ scanning electron microscope observation up to 1300 °C. I. Concept, fabrication, and validation

TERM laser-based annealing

In [9]:
# Audit all EBSD/grain-growth title hits and authors' other RSI records to identify a multi-paper pillar specialist
m=tsv[tsv.title.str.contains(r'EBSD|electron backscatter|grain growth|grain boundary|microstructure mapping',case=False,regex=True)]
print(m[['year','doi','authors','title']].sort_values('year',ascending=False).to_string(index=False))
# Search open-control candidates with thermal overlap
m2=tsv[tsv.title.str.contains(r'open.source|LabVIEW|modular data acquisition|instrument control',case=False,regex=True)]
print('\nOPEN/CONTROL HITS')
print(m2[['year','doi','authors','title']].sort_values('year',ascending=False).head(80).to_string(index=False))

 year               doi                                                                                                               authors                                                                                                                                                                                                                                                                                   title
 2022 10.1063/5.0087945 Tsai S; Konijnenberg P; Gonzalez I; Hartke S; Griffiths T; Herbig M; Kawano-Miyata K; Taniyama A; Sano N; Zaefferer S Development of a new, fully automated system for electron backscatter diffraction (EBSD)-based large volume three-dimensional microstructure mapping using serial sectioning by mechanical polishing, and its application to the analysis of special boundaries in 316L stainless steel

OPEN/CONTROL HITS
 year               doi                                                                                                                  

In [10]:
more_queries=[
'Laurent Gallais Institut Fresnel current professor laser annealing',
'Yuzhong Zhang Hefei University of Technology pyrometer current',
'G Lohöfer DLR electromagnetic levitation current',
'Saurabh Kabra ISIS neutron current',
'Sebastian Weber PyMoDAQ current affiliation',
'Andreas Bauer TUM topological materials current staff',
'Ashkan Salamat current affiliation 2025',
]
for q in more_queries:
    rr=await web_search(q,num_results=4)
    results[q]=rr
    print('\nQUERY',q)
    for x in rr[:3]: print('-',x.get('title'),x.get('url'))


QUERY Laurent Gallais Institut Fresnel current professor laser annealing
- Laurent Gallais https://www.centrale-mediterranee.fr/sites/default/files/2023-02/CV%20Laurent%20Gallais.pdf
- Laurent Gallais | Centrale Méditerranée https://www.centrale-mediterranee.fr/en/laurent-gallais
- Laurent Gallais | Centrale Méditerranée https://www.centrale-mediterranee.fr/fr/laurent-gallais



QUERY Yuzhong Zhang Hefei University of Technology pyrometer current
- Yuzhong Zhang https://yqkxen.hfut.edu.cn/2023/1121/c14386a298682/page.htm
- Development of a CCD-based pyrometer for surface temperature measurement of casting billets - IOPscience https://iopscience.iop.org/article/10.1088/1361-6501/aa6928
- 张育中 https://yqkx.hfut.edu.cn/2017/0504/c14420a298968/page.htm



QUERY G Lohöfer DLR electromagnetic levitation current
- TEMPUS—A microgravity electromagnetic levitation facility. ... https://elib.dlr.de/204312/1/TEMPUS%E2%80%94A%20microgravity%20electromagnetic%20levitation%20facility.pdf
- electronic library -  High-resolution inductive measurement of electrical resistivity and density of electromagnetically levitated liquid metal droplets https://elib.dlr.de/125740/
- EML Processing Measurement Techniques https://doi.org/10.1515/htmp.2008.27.6.389



QUERY Saurabh Kabra ISIS neutron current
- Saurabh Kabra | ORNL https://www.ornl.gov/staff-profile/saurabh-kabra
- Saurabh Kabra https://linkedin.com/in/saurabh-kabra-uk
- Engineering and Imaging techniques at ISIS neutron and ... https://www.imperial.ac.uk/events/152165/engineering-and-imaging-techniques-at-isis-neutron-and-muon-source/



QUERY Sebastian Weber PyMoDAQ current affiliation
- Sébastien Weber (@seb5g) https://github.com/seb5g
- 5. Contributors — PyMoDAQ Femto 3.1.1 documentation https://pymodaq-femto.readthedocs.io/en/latest/usage/Contributors.html
- 9. About — PyMoDAQ 5.2.0.dev220 documentation http://pymodaq.cnrs.fr/en/latest/about.html



QUERY Andreas Bauer TUM topological materials current staff
- Lehre - Chair of Experimental Physics on the Topology of Correlated Systems https://www.ph.nat.tum.de/en/tcs/lehre/
- Personen - Department of Physics https://www.ph.nat.tum.de/en/tcs/personen/
- Personen & Zuständigkeiten - TUMonline - Technische Universität München https://campus.tum.de/tumonline/wborggruppen.gruppen?pOrgNr=21569



QUERY Ashkan Salamat current affiliation 2025
- Ashkan Salamat | People | UNLV https://www.unlv.edu/people/ashkan-salamat
- Ashkan Salamat | News Center https://www.unlv.edu/news/expert/ashkan-salamat
- Ashkan Salamat - nexcl - University of Nevada, Las Vegas https://nexcl.unlv.edu/the-team/ashkan-salamat


In [11]:
# Create a transparent evidence table for the proposed core slate and verify DOI existence in supplied TSV.
slate={
'Florian Kargl':['10.1063/1.4922359','10.1063/5.0037398','10.1063/5.0151523','10.1063/5.0004356'],
'Cheng-Chi Tai':['10.1063/1.4992021','10.1063/5.0006019','10.1063/5.0066308'],
'Stephen D. Wilson':['10.1063/1.5085327','10.1063/5.0186528'],
'Jérôme Mendonça':['10.1063/5.0207466','10.1063/5.0207475','10.1063/5.0207477'],
'Yuzhong Zhang':['10.1063/1.5034233','10.1063/1.5129758','10.1063/5.0046410','10.1063/5.0101504'],
'Laurent Gallais':['10.1063/1.5133741','10.1063/5.0139508','10.1063/5.0202933'],
'Andreas Bauer':['10.1063/1.4954926','10.1063/1.4967011'],
'Ke An':['10.1063/1.4939194','10.1063/5.0299443'],
'G. Lohöfer':['10.1063/1.5065482','10.1063/5.0182719'],
'Ashkan Salamat':['10.1063/1.5040508','10.1063/5.0086642'],
'Saurabh Kabra':['10.1063/1.4937615','10.1063/1.5001085'],
'Stefan Zaefferer':['10.1063/5.0087945'],
'Sébastien Weber':['10.1063/5.0032116','10.1063/1.4978324'],
}
missing=[]
for name,dois in slate.items():
    print('\n',name)
    for d in dois:
        z=tsv[tsv.doi.eq(d)]
        if z.empty: missing.append((name,d))
        else: print(f" {int(z.iloc[0].year)} {d} | {z.iloc[0].title}")
print('\nMissing DOI records:',missing)
print('Slate size',len(slate))


 Florian Kargl
 2015 10.1063/1.4922359 | Near-isothermal furnace for in situ and real time X-ray radiography solidification experiments
 2021 10.1063/5.0037398 | Isothermal furnace for long-term in situ and real-time X-radiography solidification experiments
 2023 10.1063/5.0151523 | X-radiography front tracking gradient furnace for directional solidification of bulk Al-alloys
 2020 10.1063/5.0004356 | Gas-loading furnace for deuterium-charged alloy-casting

 Cheng-Chi Tai
 2017 10.1063/1.4992021 | Magnetic induction of hyperthermia by a modified self-learning fuzzy temperature controller
 2020 10.1063/5.0006019 | A modified self-tuning fuzzy logic temperature controller for metal induction heating
 2022 10.1063/5.0066308 | Liver tumor ablation enhancement by induction-heating system with bitter-like deep magnetic field coil

 Stephen D. Wilson
 2019 10.1063/1.5085327 | High-pressure laser floating zone furnace
 2024 10.1063/5.0186528 | Advances in high-pressure laser floating zone gro

In [12]:
# Final targeted public-web COI search: candidate names against manuscript authors and flagged institutions.
from e14c.web import web_search
coi_queries=[
'("Florian Kargl" OR "Cheng-Chi Tai" OR "Stephen D. Wilson" OR "Jérôme Mendonça") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")',
'("Yuzhong Zhang" OR "Laurent Gallais" OR "Andreas Bauer" OR "Ke An") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")',
'("G. Lohöfer" OR "Ashkan Salamat" OR "Saurabh Kabra" OR "Stefan Zaefferer" OR "Sébastien Weber") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")'
]
for q in coi_queries:
    rr=await web_search(q,num_results=8)
    print('\nQUERY',q)
    for x in rr:
        print('-',x.get('title'),x.get('url'))


QUERY ("Florian Kargl" OR "Cheng-Chi Tai" OR "Stephen D. Wilson" OR "Jérôme Mendonça") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")
- Sterling Baird https://acceleration.utoronto.ca/people/sterling-baird
- AccelerationConsortium (Acceleration Consortium) https://huggingface.co/AccelerationConsortium
- Interleaved bond and magnetic frustration in triangular lattice ... https://arxiv.org/html/2501.04203v1
- Erick Lawrence https://orcid.org/0000-0003-0819-1895
- Ilija Zeljkovic https://scholar.google.com/citations?hl=en&user=1I_ojZcAAAAJ
- Meet the BYU AI Committee https://genai.byu.edu/meet-the-committee
- About https://siliconslopes.byu.edu/about
- Investigators | Psychiatry | U of U School of Medicine https://medicine.utah.edu/psychiatry/research/labs/apr/investigators



QUERY ("Yuzhong Zhang" OR "Laurent Gallais" OR "Andreas Bauer" OR "Ke An") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")
- From a lab automation infrastructure perspective, the A-Lab from Gerbrand Ceder's lab is incredible!😲 It's on par with labs from Jason Hein, Andrew Cooper, Ian Foster, Helge S. Stein, Xuefeng Yu, and … | Sterling G. Baird https://www.linkedin.com/posts/sterling-baird_from-a-lab-automation-infrastructure-perspective-activity-7136432975298117632-eXss
- Ke An, Senior Scientist, Ph.D. Materials and Engineering ... https://www.ornl.gov/sites/default/files/2021-06/KeAnCV_2021_June.pdf
- Distinctive Style in Ravel's Piano Concerto in G Major https://www.front-sci.com/journal/article?doi=10.32629%2Fasc.v5i6.3264
- Adapting arepo-rt for exascale computing: GPU acceleration and ... https://inspirehep.net/literature/2781706
- Zhang, Yacong | BibSonomy https://www.bibsonomy.org/person/1c673f4cfc41c5d04672e81


QUERY ("G. Lohöfer" OR "Ashkan Salamat" OR "Saurabh Kabra" OR "Stefan Zaefferer" OR "Sébastien Weber") ("Sterling Baird" OR "Oliver Johnson" OR "Ryan Weber" OR BYU OR "University of Utah" OR "Acceleration Consortium")
- sterling.baird@utah.edu maintainer information - Repology https://repology.org/maintainer/sterling.baird%40utah.edu
- New Acceleration Consortium at University of Toronto applies ... https://www.artsci.utoronto.ca/news/new-acceleration-consortium-university-toronto-applies-artificial-intelligence-discovery
- CSD 1834614: Experimental Crystal Structure Determination https://doi.org/10.25505/fiz.icsd.cc1zl23r
- CSD 1962857: Experimental Crystal Structure Determination https://doi.org/10.25505/fiz.icsd.cc23whzh
- CSD 1784259: Experimental Crystal Structure Determination https://doi.org/10.25505/fiz.icsd.cc1xwnr6
- ashkan salamat – Frank's World of Data Science & AI https://www.franksworld.com/tag/ashkan-salamat/
- Characterisation of a Fe(gamma)-Ni-Fe(alpha) Multi-materia